# 🤖 SmolVLA GPU 訓練（防斷線 / 斷線自動續訓版）

**為什麼 Colab 會中斷？**
- 免費版 Colab 有 **閒置逾時**（約 90 分鐘沒有操作頁面就會斷開）與 **單次執行上限**（最長約 12 小時，GPU 忙碌時更短），以及 **每日 GPU 配額**。
- 這些是 Google 的平台限制，**任何 notebook 寫法都無法 100% 保證不斷線**。

**本筆記本的做法：讓斷線「不會損失進度」**
1. 🧵 **背景執行訓練**（`nohup`）：關掉分頁、中斷儲存格、瀏覽器斷線都不會停止訓練程序（只要執行階段 VM 還活著）。
2. 💾 **每次存檔自動同步到 Google Drive**：背景同步程式在每個 Checkpoint（含 optimizer / scheduler / step）寫好後，以「原子方式」複製到雲端硬碟，只保留最新一份節省空間。
3. 🔁 **真正的續訓（`--resume=true`）**：重新連線後「全部執行」，會自動從 Drive 上最新 Checkpoint 的 **同一步數、同一學習率排程** 繼續，直到達到 `TOTAL_STEPS`。

> 斷線後只要：重新連線 → 選 GPU → **執行階段 → 全部執行**，就會自動接續。

⚠️ 請先：**執行階段 → 變更執行階段類型 → T4 GPU**

In [ ]:
# 1. 驗證 GPU 並取得 LeRobot 程式碼
import os, torch

if not torch.cuda.is_available():
    raise SystemError("❌ 未檢測到 GPU！請至 [執行階段 -> 變更執行階段類型] 切換至 T4 GPU 後再執行。")
print("✅ GPU:", torch.cuda.get_device_name(0),
      f"({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")

if not os.path.exists("/content/lerobot"):
    os.system("git clone https://github.com/Child-pi/lerobot.git /content/lerobot")
else:
    os.system("cd /content/lerobot && git pull origin main")
%cd /content/lerobot

### ⚙️ 步驟 2：訓練設定與 Google Drive 掛載
- `TOTAL_STEPS`：**整個訓練的總目標步數**（不是本次步數）。斷線續訓會一直跑到這個數字為止。
- `SAVE_FREQ`：多少步存一次檔並同步到 Drive。越小斷線損失越少，但存檔較頻繁（每份約數 GB）。
- `AUTO_RESUME`：勾選時自動從 Drive 最新 Checkpoint 續訓；取消勾選則從 `lerobot/smolvla_base` 重新開始。

In [ ]:
#@title 訓練設定 (Training Settings)
TOTAL_STEPS = 20000 #@param {type:"integer"}
SAVE_FREQ = 500 #@param {type:"integer"}
BATCH_SIZE = 2 #@param {type:"integer"}
AUTO_RESUME = True #@param {type:"boolean"}
DATASET_REPO_ID = "lerobot/aloha_sim_insertion_human" #@param ["lerobot/aloha_sim_insertion_human", "lerobot/aloha_sim_transfer_cube_human"]
DRIVE_ROOT = "/content/drive/MyDrive/SmolVLA_training" #@param {type:"string"}

import os
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

# 與舊版筆記本相容：/content/data/SmolVLA_training -> Drive
os.makedirs("/content/data", exist_ok=True)
if not os.path.exists("/content/data/SmolVLA_training"):
    try:
        os.symlink(DRIVE_ROOT, "/content/data/SmolVLA_training")
    except OSError:
        pass

LEROBOT_DIR = "/content/lerobot"
OUTPUT_DIR = os.path.join(LEROBOT_DIR, "outputs/train/smolvla_aloha")
LOCAL_CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
DRIVE_CKPT = os.path.join(DRIVE_ROOT, "checkpoint_latest")   # 最新 Checkpoint（含 training_state）
LOG_FILE = "/content/train_log.txt"                          # 日誌寫在本機（Drive 掛載會緩衝寫入，無法即時讀取）
DRIVE_LOG_FILE = os.path.join(DRIVE_ROOT, "train_log.txt")   # 背景同步程式會定期複製一份到 Drive
PID_FILE = "/content/train.pid"

def train_running():
    # 檢查背景訓練程序是否仍在執行（已結束但未回收的 zombie 程序視為已結束）
    try:
        pid = int(open(PID_FILE).read().strip())
        with open(f"/proc/{pid}/status") as fh:
            for line in fh:
                if line.startswith("State:"):
                    return "Z" not in line and "X" not in line
        return True
    except Exception:
        return False

print("✅ Drive 已掛載:", DRIVE_ROOT)
print(f"🎯 目標總步數: {TOTAL_STEPS}，每 {SAVE_FREQ} 步存檔並同步到 Drive")

### 📦 步驟 3：安裝依賴（每次新的執行階段都需要重跑）

In [ ]:
%cd /content/lerobot
!pip install -q -e ".[smolvla]"
!pip install -q dm_env dm_tree glfw lxml pyopengl pyparsing
!pip install -q --upgrade "mujoco>=3.15.0"
!pip install -q dm-control --no-deps
!pip install -q gym-aloha --no-deps
!pip install -q av
print("🎉 環境安裝完成！")

### 🔁 步驟 4：從 Google Drive 還原最新 Checkpoint
- 找到本筆記本產生的 `checkpoint_latest/` → **真正續訓**（步數、optimizer、學習率排程完全接續）。
- 只找到舊版筆記本的權重 → 用它當起點 **重新開始排程的微調**（舊版只跑 500 步，學習率排程已衰減完，不適合直接 resume）。
- 都沒有 → 從 `lerobot/smolvla_base` 開始。

In [ ]:
import os, glob, shutil, json

RESUME_MODE = "fresh"        # "resume" | "warm_start" | "fresh"
WARM_START_PATH = None
resume_step = 0

def _read_step(ckpt_dir):
    p = os.path.join(ckpt_dir, "training_state", "training_step.json")
    if os.path.exists(p):
        with open(p) as f:
            return int(json.load(f).get("step", 0))
    return int(os.path.basename(ckpt_dir)) if os.path.basename(ckpt_dir).isdigit() else 0

step_file = os.path.join(DRIVE_CKPT, "STEP")
if AUTO_RESUME and os.path.exists(step_file):
    step_name = open(step_file).read().strip()
    src = os.path.join(DRIVE_CKPT, step_name)
    dst = os.path.join(LOCAL_CKPT_DIR, step_name)
    if os.path.isdir(src):
        if os.path.exists(OUTPUT_DIR):
            shutil.rmtree(OUTPUT_DIR)
        print(f"⏬ 從 Drive 還原 Checkpoint {step_name}（數 GB，約需 1~3 分鐘）...")
        shutil.copytree(src, dst)
        os.symlink(step_name, os.path.join(LOCAL_CKPT_DIR, "last"))
        resume_step = _read_step(dst)
        RESUME_MODE = "resume"
        print(f"✅ 將從第 {resume_step} 步真正續訓（目標 {TOTAL_STEPS} 步）")
elif AUTO_RESUME:
    legacy = sorted(glob.glob(os.path.join(DRIVE_ROOT, "outputs/train/*/checkpoints/*/pretrained_model")),
                    key=os.path.getmtime)
    if legacy:
        WARM_START_PATH = "/content/warm_start_model"
        shutil.rmtree(WARM_START_PATH, ignore_errors=True)
        shutil.copytree(legacy[-1], WARM_START_PATH)
        RESUME_MODE = "warm_start"
        print("♻️ 找到舊版權重，將作為起點重新微調:", legacy[-1])

if RESUME_MODE == "fresh":
    print("🆕 從 lerobot/smolvla_base 全新開始訓練")
    if os.path.exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)

### 🚀 步驟 5：在背景啟動訓練 + Drive 自動同步
此儲存格會**立即結束**，訓練在背景持續進行。請用下一格「步驟 6」觀察進度。

In [ ]:
import os, subprocess, shlex, time

SYNC_SCRIPT = "/content/sync_ckpt.py"
with open(SYNC_SCRIPT, "w") as f:
    f.write('''
import os, sys, time, shutil
local_dir, drive_dst, pid_file, log_file, drive_log = sys.argv[1:6]
synced = None

def train_alive():
    try:
        pid = int(open(pid_file).read().strip())
        with open(f"/proc/{pid}/status") as fh:
            for line in fh:
                if line.startswith("State:"):
                    return "Z" not in line and "X" not in line
        return True
    except Exception:
        return False

def sync_once():
    global synced
    link = os.path.join(local_dir, "last")
    if not os.path.islink(link):
        return
    target = os.path.realpath(link)
    step = os.path.basename(target)
    if step == synced or not os.path.isdir(target):
        return
    time.sleep(5)
    tmp, old = drive_dst + ".tmp", drive_dst + ".old"
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(target, os.path.join(tmp, step))
    with open(os.path.join(tmp, "STEP"), "w") as fh:
        fh.write(step)
    shutil.rmtree(old, ignore_errors=True)
    if os.path.exists(drive_dst):
        os.rename(drive_dst, old)
    os.rename(tmp, drive_dst)          # 原子替換：Drive 上永遠有一份完整 Checkpoint
    shutil.rmtree(old, ignore_errors=True)
    synced = step
    print(time.strftime("%H:%M:%S"), "☁️ 已同步 Checkpoint", step, "到 Drive", flush=True)
    for d in os.listdir(local_dir):     # 本機只保留最新一份，避免磁碟爆滿
        p = os.path.join(local_dir, d)
        if d not in ("last", step) and os.path.isdir(p) and not os.path.islink(p):
            shutil.rmtree(p, ignore_errors=True)

while True:
    alive = train_alive()
    try:
        sync_once()
    except Exception as e:
        print("⚠️ Checkpoint 同步失敗，稍後重試:", e, flush=True)
    try:
        shutil.copyfile(log_file, drive_log)   # 日誌備份到 Drive，斷線後也能查看
    except Exception:
        pass
    if not alive:
        break
    time.sleep(30)
print("✅ 同步程式結束", flush=True)
''')

if train_running():
    print("ℹ️ 訓練已經在背景執行中，請直接執行步驟 6 觀察進度。")
elif RESUME_MODE == "resume" and resume_step >= TOTAL_STEPS:
    print(f"🏁 已完成 {resume_step} 步 ≥ TOTAL_STEPS={TOTAL_STEPS}。若要再訓練，請在步驟 2 調高 TOTAL_STEPS。")
else:
    common = [
        f"--steps={TOTAL_STEPS}",
        f"--save_freq={SAVE_FREQ}",
        f"--batch_size={BATCH_SIZE}",
        f"--output_dir={OUTPUT_DIR}",
        "--log_freq=20",
        "--wandb.enable=false",
        "--policy.push_to_hub=false",
    ]
    if RESUME_MODE == "resume":
        cmd = ["lerobot-train",
               f"--config_path={os.path.join(LOCAL_CKPT_DIR, 'last/pretrained_model/train_config.json')}",
               "--resume=true"] + common
    else:
        cmd = ["lerobot-train",
               f"--policy.path={WARM_START_PATH or 'lerobot/smolvla_base'}",
               f"--dataset.repo_id={DATASET_REPO_ID}",
               "--policy.empty_cameras=2",
               '--rename_map={"observation.images.top": "observation.images.camera1"}'] + common

    shell_cmd = (
        f"cd {LEROBOT_DIR} && MUJOCO_GL=egl PYTHONUNBUFFERED=1 "
        f"nohup {' '.join(shlex.quote(c) for c in cmd)} >> {shlex.quote(LOG_FILE)} 2>&1 & echo $! > {PID_FILE}"
    )
    with open(LOG_FILE, "a") as f:
        f.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} 啟動 ({RESUME_MODE}) =====\n")
    subprocess.run(shell_cmd, shell=True, check=True)
    time.sleep(3)
    subprocess.run(
        f"nohup python {SYNC_SCRIPT} {shlex.quote(LOCAL_CKPT_DIR)} {shlex.quote(DRIVE_CKPT)} {PID_FILE} "
        f"{shlex.quote(LOG_FILE)} {shlex.quote(DRIVE_LOG_FILE)} "
        f">> /content/sync_log.txt 2>&1 &",
        shell=True, check=True)
    print("🚀 訓練已在背景啟動！PID =", open(PID_FILE).read().strip())
    print("📄 日誌:", LOG_FILE)
    print("👉 請執行步驟 6 觀察即時進度（中斷步驟 6 不會停止訓練）")

### 📈 步驟 6：觀察訓練進度（會自動結束）
- 訓練會跑好幾個小時，這格**不需要一直等到訓練結束**：預設觀察 `MONITOR_MINUTES` 分鐘後自動結束，**訓練仍在背景繼續**。
- 隨時按 ■ 停止此格也不會影響訓練；想再看進度就重新執行這一格。
- 訓練完成或發生錯誤時，這格會立即結束並顯示結果。
- 若要**真的停止訓練**，請執行最下方的「🛑 停止訓練」儲存格。

In [ ]:
#@title 觀察訓練進度
MONITOR_MINUTES = 10 #@param {type:"integer"}
import os, time, re

_PROG = re.compile(r"Training:\s*\d+%.*?\|\s*(\d+)/(\d+)\s*\[([^\]<]*)<([^\],]*)")

def _status(text):
    m = None
    for m in _PROG.finditer(text):
        pass
    if m:
        done, total, elapsed, remain = int(m.group(1)), int(m.group(2)), m.group(3).strip(), m.group(4).strip()
        return f"進度 {done}/{total} ({done / max(total, 1) * 100:.1f}%) | 已用 {elapsed} | 預估剩餘 {remain}"
    return "啟動中（載入資料集 / 模型，首次約需數分鐘）"

if not os.path.exists(LOG_FILE):
    print("⚠️ 找不到日誌，請先執行步驟 5 啟動訓練。")
else:
    pos = max(0, os.path.getsize(LOG_FILE) - 3000)
    deadline = time.time() + MONITOR_MINUTES * 60
    last_status_t = 0
    tail = ""
    try:
        while True:
            with open(LOG_FILE, errors="ignore") as f:
                f.seek(pos)
                chunk = f.read()
                pos = f.tell()
            tail = (tail + chunk)[-20000:]
            for line in re.split(r"[\r\n]+", chunk):
                s = line.strip()
                if s and "Training:" not in s and "check_worker_number_rationality" not in s \
                        and "Detected no triton" not in s:
                    print(s, flush=True)

            running = train_running()
            if time.time() - last_status_t >= 60 or not running:
                print(f"⏳ [{time.strftime('%H:%M:%S')}] {_status(tail)}", flush=True)
                last_status_t = time.time()

            if not running:
                if "Traceback" in tail or "Error" in tail.split("啟動")[-1][-3000:]:
                    print("\n❌ 訓練程序已結束，日誌中有錯誤訊息，請查看上方內容。")
                else:
                    print("\n🏁 訓練程序已結束（完成）。可以執行步驟 7 進行評估。")
                break
            if time.time() >= deadline:
                print(f"\n✅ 已觀察 {MONITOR_MINUTES} 分鐘，此格自動結束；訓練仍在背景執行中。")
                print("👉 想再看進度，重新執行這一格即可。日誌也會同步到:", DRIVE_LOG_FILE)
                break
            time.sleep(10)
    except KeyboardInterrupt:
        print("\n👋 已停止觀察；訓練仍在背景執行。")

### 🎬 步驟 7：模擬評估並錄製影片（訓練結束後執行）

In [ ]:
import os, glob, subprocess

ckpt = os.path.join(LOCAL_CKPT_DIR, "last", "pretrained_model")
if not os.path.exists(ckpt) and os.path.exists(os.path.join(DRIVE_CKPT, "STEP")):
    step_name = open(os.path.join(DRIVE_CKPT, "STEP")).read().strip()
    ckpt = os.path.join(DRIVE_CKPT, step_name, "pretrained_model")

if not os.path.exists(ckpt):
    print("❌ 找不到可評估的 Checkpoint")
else:
    print("✅ 評估模型:", ckpt)
    eval_cmd = [
        "lerobot-eval",
        f"--policy.path={ckpt}",
        "--env.type=aloha",
        "--env.task=" + ("AlohaTransferCube-v0" if "transfer_cube" in DATASET_REPO_ID else "AlohaInsertion-v0"),
        "--eval.n_episodes=3",
        "--eval.batch_size=1",
        "--eval.use_async_envs=false",
        '--rename_map={"observation.images.top": "observation.images.camera1"}',
        "--output_dir=/content/lerobot/outputs/eval/smolvla_aloha",
    ]
    env = dict(os.environ, MUJOCO_GL="egl", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(eval_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in iter(p.stdout.readline, ""):
        if "Running rollout" not in line:
            print(line, end="", flush=True)
    p.wait()
    print("✅ 評估完成" if p.returncode == 0 else f"⚠️ 評估結束，狀態碼 {p.returncode}")

In [ ]:
# 📺 播放影片並備份評估結果到 Drive
import glob, os, shutil
from IPython.display import Video, display

videos = sorted(glob.glob("/content/lerobot/outputs/eval/**/*.mp4", recursive=True), key=os.path.getmtime)
if videos:
    print("🎥", videos[-1])
    display(Video(videos[-1], embed=True, width=640))
    shutil.copytree("/content/lerobot/outputs/eval", os.path.join(DRIVE_ROOT, "eval"), dirs_exist_ok=True)
    print("☁️ 評估影片已備份到", os.path.join(DRIVE_ROOT, "eval"))
else:
    print("⚠️ 尚未找到影片")

### 🛑 停止訓練（需要時才執行）
停止後 Drive 上仍保留最後一次同步的 Checkpoint，之後可再「全部執行」續訓。

In [ ]:
import os, signal
try:
    pid = int(open(PID_FILE).read().strip())
    os.kill(pid, signal.SIGTERM)
    print("🛑 已送出停止訊號給訓練程序 PID", pid)
except Exception as e:
    print("ℹ️ 沒有正在執行的訓練:", e)